viruses embedded in book files are relatively rare, especially if you're dealing with legitimate .pdf or .epub files. Malware in books is more commonly associated with certain scenarios, such as:

How Books Can Carry Viruses
Macros in Documents:

Malicious macros can be embedded in .doc, .docx, or .pdf files. When opened in specific applications, they can execute harmful code.
For PDFs, this could mean embedded JavaScript, which can be used maliciously.
Executable Payloads in Archives:

Sometimes, books are distributed in .zip or .rar files containing additional hidden executables.
Trojanized eBook Readers:

The malware may not reside in the book itself but in the reader app (or a fake version of it).
Phishing Links in Metadata or Content:

Malicious links may be hidden within the book content or metadata. If clicked, they can lead to phishing sites or malware downloads.
Tampered File Types:

A file disguised as a .pdf but actually an .exe or .bat file can execute harmful scripts when opened.

Outline of the Program

1. Scan a Directory: Find all book files in a given folder and its subfolders.


2. Extract Metadata: Use libraries to read metadata from the files.


3. Analyze Metadata: Check for potentially suspicious metadata, such as unusually large or strange entries.


4. Clean Metadata: Remove or modify problematic entries.


5. Save Changes: Write the cleaned metadata back to the file.

In [ ]:

import os
import shutil
from PyPDF2 import PdfReader, PdfWriter
from ebooklib import epub
from ebooklib.epub import EpubBook
import subprocess

def scan_directory(directory, extensions):
    """
    Scan a directory and return a list of files with the given extensions.
    """
    files = []
    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            if any(filename.lower().endswith(ext) for ext in extensions):
                files.append(os.path.join(root, filename))
    return files

def process_pdf(file_path):
    """
    Extract and clean metadata for a PDF file.
    """
    try:
        reader = PdfReader(file_path)
        writer = PdfWriter()

        # Extract metadata
        metadata = reader.metadata
        print(f"Original metadata for {file_path}: {metadata}")

        # Clean metadata
        cleaned_metadata = {}
        for key, value in metadata.items():
            if isinstance(value, str) and len(value) < 1000:  # Filter unusually long entries
                cleaned_metadata[key] = value

        # Write the new metadata to a new PDF
        for page in reader.pages:
            writer.add_page(page)
        writer.add_metadata(cleaned_metadata)

        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original
        with open(file_path, "wb") as f:
            writer.write(f)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def process_epub(file_path):
    """
    Extract and clean metadata for an EPUB file.
    """
    try:
        book = epub.read_epub(file_path)
        metadata = book.get_metadata("DC", "*")
        print(f"Original metadata for {file_path}: {metadata}")

        # Clean metadata
        cleaned_metadata = []
        for meta in metadata:
            key, value, _ = meta
            if isinstance(value, str) and len(value) < 1000:  # Filter unusually long entries
                cleaned_metadata.append((key, value, {}))

        # Set cleaned metadata
        book.set_metadata("DC", cleaned_metadata)

        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original
        epub.write_epub(file_path, book)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def process_other_files_with_exiftool(file_path):
    """
    Use ExifTool to clean metadata for other file types.
    """
    try:
        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original

        # Clean metadata using exiftool
        subprocess.run(["exiftool", "-all=", "-overwrite_original", file_path], check=True)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def main():
    directory = input("Enter the directory containing your books: ")
    extensions = [".pdf", ".epub", ".mobi"]
    files = scan_directory(directory, extensions)

    for file_path in files:
        if file_path.lower().endswith(".pdf"):
            process_pdf(file_path)
        elif file_path.lower().endswith(".epub"):
            process_epub(file_path)
        else:
            process_other_files_with_exiftool(file_path)

if __name__ == "__main__":
    main()

Key Features of the Script
Selective Metadata Cleaning: Retains essential fields (Title, Author, Subject, etc.) and removes only suspicious or unnecessary entries.
No Backups: The script directly modifies files without creating backups.
File Extension Flexibility: Handles .pdf and .epub formats while being extensible for other types.
Logging: Logs actions to a file for review, ensuring transparency about what metadata was removed or retained.

In [1]:
import os
from PyPDF2 import PdfReader, PdfWriter
from ebooklib import epub

def scan_directory(directory, extensions):
    """
    Scan a directory and return a list of files with the given extensions.
    """
    files = []
    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            if any(filename.lower().endswith(ext) for ext in extensions):
                files.append(os.path.join(root, filename))
    return files

def process_pdf(file_path):
    """
    Clean metadata for a PDF file while retaining essential fields.
    """
    try:
        reader = PdfReader(file_path)
        writer = PdfWriter()

        # Extract metadata
        metadata = reader.metadata
        print(f"[INFO] Original metadata for {file_path}: {metadata}")

        # Define essential metadata fields to retain
        essential_fields = {"/Title", "/Author", "/Subject", "/Keywords"}
        cleaned_metadata = {key: value for key, value in metadata.items() if key in essential_fields}

        # Write the new metadata and pages to the PDF
        for page in reader.pages:
            writer.add_page(page)
        writer.add_metadata(cleaned_metadata)

        # Save changes to the same file
        with open(file_path, "wb") as f:
            writer.write(f)
        print(f"[SUCCESS] Metadata cleaned for {file_path}")
    except Exception as e:
        print(f"[ERROR] Could not process {file_path}: {e}")

def process_epub(file_path):
    """
    Clean metadata for an EPUB file while retaining essential fields.
    """
    try:
        book = epub.read_epub(file_path)
        metadata = book.get_metadata("DC", "*")
        print(f"[INFO] Original metadata for {file_path}: {metadata}")

        # Define essential fields to retain (e.g., title, creator)
        essential_fields = {"title", "creator", "subject", "date", "description"}
        cleaned_metadata = [
            (key, value, attributes)
            for key, value, attributes in metadata
            if key.lower() in essential_fields and isinstance(value, str) and len(value) < 1000
        ]

        # Update metadata in the EPUB
        book.set_metadata("DC", cleaned_metadata)
        epub.write_epub(file_path, book)
        print(f"[SUCCESS] Metadata cleaned for {file_path}")
    except Exception as e:
        print(f"[ERROR] Could not process {file_path}: {e}")

def main():
    # Specify the directory containing books
    directory = input("Enter the directory containing your books: ").strip()
    if not os.path.isdir(directory):
        print("[ERROR] Invalid directory. Please specify a valid path.")
        return

    # Define file extensions to process
    extensions = [".pdf", ".epub"]

    # Scan directory for supported files
    files = scan_directory(directory, extensions)
    if not files:
        print("[INFO] No files found to process.")
        return

    print(f"[INFO] Found {len(files)} file(s) to process.")

    # Process each file based on its type
    for file_path in files:
        print(f"[INFO] Processing {file_path}...")
        if file_path.lower().endswith(".pdf"):
            process_pdf(file_path)
        elif file_path.lower().endswith(".epub"):
            process_epub(file_path)

if __name__ == "__main__":
    main()


[INFO] Found 366 file(s) to process.
[INFO] Processing H:\My Drive\Books\Поехали! 1.pdf...
[INFO] Original metadata for H:\My Drive\Books\Поехали! 1.pdf: {'/Producer': 'GPL Ghostscript 8.70', '/CreationDate': "D:20100117145304+01'00'", '/ModDate': "D:20100117145304+01'00'"}
[SUCCESS] Metadata cleaned for H:\My Drive\Books\Поехали! 1.pdf
[INFO] Processing H:\My Drive\Books\Colloquial Russian 2.pdf...
[INFO] Original metadata for H:\My Drive\Books\Colloquial Russian 2.pdf: {'/Producer': 'Acrobat Distiller 6.0 (Windows)', '/CreationDate': "D:20060113144625+05'30'", '/Author': 'Svetlana Le Flemming and Susan E.Kay', '/ModDate': "D:20080203233824-06'00'", '/Title': 'Colloquial Russian: The Next Step in Language Learning'}
[SUCCESS] Metadata cleaned for H:\My Drive\Books\Colloquial Russian 2.pdf
[INFO] Processing H:\My Drive\Books\Living Russian Grammar.pdf...
[INFO] Original metadata for H:\My Drive\Books\Living Russian Grammar.pdf: {'/Producer': 'ScanKromsator', '/ModDate': "D:201106211140

PdfReadError("Invalid Elementary Object starting with b'U' @37825790: b'rants<</PANTONE 293 U 4716 0 R>>>>endobj\\n4718 0 obj<</Domain[0 1]/N 1/Range[0 1 '")
Multiple definitions in dictionary at byte 0x2412d55 for key 0
Multiple definitions in dictionary at byte 0x2412d59 for key 0
PdfReadError("Invalid Elementary Object starting with b']' @37825861: b'ange[0 1 0 1 0 1 0 1]/C1[0.91 0.68 0 0]/C0[0 0 0 0]/FunctionType 2>>endobj\\n4719 '")
PdfReadError("Invalid Elementary Object starting with b'C' @38064577: b'rants<</PANTONE 293 CV 4901 0 R>>>>endobj\\n4900 0 obj[/DeviceN[/PANTONE#20293#20C'")
PdfReadError("Invalid Elementary Object starting with b'V' @38064637: b'N[/PANTONE#20293#20CV/None/None/None]4902 0 R 4903 0 R 4899 0 R]endobj\\n4901 0 ob'")
PdfReadError("Invalid Elementary Object starting with b'C' @38065124: b'rants<</PANTONE 293 CV 4907 0 R>>>>endobj\\n4906 0 obj<</Domain[0 1]/N 1/Range[0 1'")
Multiple definitions in dictionary at byte 0x244d43c for key 0
Multiple definitions 

[SUCCESS] Metadata cleaned for H:\My Drive\Books\Pritchard-Fox-McDonalds_2011_8ed_Fluid-Mechanics.pdf
[INFO] Processing H:\My Drive\Books\Pierson_ Brown_ Easton_ Howard_ Pinder - Business Finance (0, McGrawHill).pdf...
[INFO] Original metadata for H:\My Drive\Books\Pierson_ Brown_ Easton_ Howard_ Pinder - Business Finance (0, McGrawHill).pdf: {'/Producer': 'ABBYY FineReader 12', '/CreationDate': 'D:20170228051641Z', '/ModDate': "D:20170814184901+10'00'"}
[SUCCESS] Metadata cleaned for H:\My Drive\Books\Pierson_ Brown_ Easton_ Howard_ Pinder - Business Finance (0, McGrawHill).pdf
[INFO] Processing H:\My Drive\Books\AnimalFarm.pdf...
[INFO] Original metadata for H:\My Drive\Books\AnimalFarm.pdf: {'/Producer': 'htmldoc 1.8.11 Copyright 1997-2001 Easy Software Products, All Rights Reserved.', '/CreationDate': 'D:20011015173048Z', '/Title': ' Animal Farm', '/Author': 'George Orwell'}
[SUCCESS] Metadata cleaned for H:\My Drive\Books\AnimalFarm.pdf
[INFO] Processing H:\My Drive\Books\2017_Boo

c:\Users\ilo_l\AppData\Local\Programs\Python\Python312\Lib\site-packages\ebooklib\epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
c:\Users\ilo_l\AppData\Local\Programs\Python\Python312\Lib\site-packages\ebooklib\epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


[SUCCESS] Metadata cleaned for H:\My Drive\Books\Akhlesh Lakhtakia and Raúl José Martín-Palma (Eds.) - Engineered Biomimicry (2013).pdf
[INFO] Processing H:\My Drive\Books\Kevin M. Passino (auth.) - Biomimicry for Optimization, Control, and Automation-Springer-Verlag London (2005).pdf...
[INFO] Original metadata for H:\My Drive\Books\Kevin M. Passino (auth.) - Biomimicry for Optimization, Control, and Automation-Springer-Verlag London (2005).pdf: {'/ModDate': "D:20130715141830+01'00'", '/Creator': 'PdfCompressor 3.1.34', '/Producer': 'CVISION Technologies', '/CreationDate': "D:20130630085623+04'00'", '/Author': '\x104<8=8AB@0B>@'}
[SUCCESS] Metadata cleaned for H:\My Drive\Books\Kevin M. Passino (auth.) - Biomimicry for Optimization, Control, and Automation-Springer-Verlag London (2005).pdf
[INFO] Processing H:\My Drive\Books\Baumeister, Dayna_ Benyus, Janine M._ Dwyer, Jamie_ Ritter, Sherry_ Tocke, Rose - Biomimicry _ resource handbook _ a seed bank of best practices-CreateSpace (2011

...

[ERROR] Could not process H:\My Drive\Books\William W. Lytton - From computer to brain_ foundations of computational neuroscience-Springer (2002).pdf: maximum recursion depth exceeded while calling a Python object
[INFO] Processing H:\My Drive\Books\zapiski pokoinika ws - Mikhail Bulghakov.epub...
[ERROR] Could not process H:\My Drive\Books\zapiski pokoinika ws - Mikhail Bulghakov.epub: 'Bad Zip file'
[INFO] Processing H:\My Drive\Books\Барсуков В.И. - Электромагнетизм. Постоянный ток и его за...но...еле в вакууме и в веществе_ Конспект лекций-Издательство ТГТУ (2007).pdf...
[ERROR] Could not process H:\My Drive\Books\Барсуков В.И. - Электромагнетизм. Постоянный ток и его за...но...еле в вакууме и в веществе_ Конспект лекций-Издательство ТГТУ (2007).pdf: EOF marker not found
[INFO] Processing H:\My Drive\Books\Владимир Сергеевич Ефремов - Достоевский_ психиатрия и литература-Диалект (2006).pdf...
[ERROR] Could not process H:\My Drive\Books\Владимир Сергеевич Ефремов - Достоевский_ псих

incorrect startxref pointer(1)


[INFO] Original metadata for H:\My Drive\Books\Л.Н. Толстой - Полное собрание сочинений. Война и мир. Т.4. Том 12-«Художественная литература» (1940).pdf: {'/Producer': 'GNU Ghostscript 7.05', '/Title': 'Joseph Heller - Catch - 22 - Notepad', '/Creator': 'Windows NT 4.0', '/CreationDate': '8:46 2/17/2003'}
[ERROR] Could not process H:\My Drive\Books\Л.Н. Толстой - Полное собрание сочинений. Война и мир. Т.4. Том 12-«Художественная литература» (1940).pdf: '/Kids'
[INFO] Processing H:\My Drive\Books\Мила Левчук.epub...
[ERROR] Could not process H:\My Drive\Books\Мила Левчук.epub: 'Bad Zip file'
[INFO] Processing H:\My Drive\Books\Нечаев Александр. Илья Муромец - royallib.com.epub...
[ERROR] Could not process H:\My Drive\Books\Нечаев Александр. Илья Муромец - royallib.com.epub: 'Bad Zip file'
[INFO] Processing H:\My Drive\Books\ПГС ракет морского базирования.pdf...
[ERROR] Could not process H:\My Drive\Books\ПГС ракет морского базирования.pdf: EOF marker not found
[INFO] Processing H:\My D

incorrect startxref pointer(2)
Object 143 0 not defined.


[ERROR] Could not process H:\My Drive\Books\Перельман_Я.И._Занимательная_физика._Книга_1_(1913).pdf: negative seek value -1
[INFO] Processing H:\My Drive\Books\Першенков П.П., Уфельман Н.А. - Электромагнетизм. Электромагнитные колебания и волны_ Методические указания-Изд-во ПГУ (2009).pdf...
[INFO] Original metadata for H:\My Drive\Books\Першенков П.П., Уфельман Н.А. - Электромагнетизм. Электромагнитные колебания и волны_ Методические указания-Изд-во ПГУ (2009).pdf: {'/Creator': 'Apache FOP Version svn-trunk', '/CreationDate': "D:20200219145635+03'00'", '/Producer': 'Apache FOP Version svn-trunk', '/Subject': 'religion_esoterics', '/Author': 'Лариса Ренар', '/Title': 'Круг женской силы. Энергии стихий и тайны обольщения'}
[ERROR] Could not process H:\My Drive\Books\Першенков П.П., Уфельман Н.А. - Электромагнетизм. Электромагнитные колебания и волны_ Методические указания-Изд-во ПГУ (2009).pdf: 'NoneType' object has no attribute 'get_object'
[INFO] Processing H:\My Drive\Books\Полное ру

In [1]:
pip install PyPDF2 ebooklib

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for ebooklib: filename=EbookLib-0.18-py3-none-any.whl size=38805 sha256=e682392dc3ec33f658dfe6c6acfd1ae74f3aefb56cd2ceec2bc3890fb74029fc
  Stored in directory: c:\users\ilo_l\appdata\local\pip\cache\wheels\c9\95\88\28e51b74669d4e3df4e72e2a031ea6c211a93a5008284d6c67
Successfully built ebooklib
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Specify your directory or file here
file_or_directory = "H:\My Drive\Books"

# Supported extensions
extensions = [".pdf", ".epub", ".mobi"]

# Scan for files if a directory is specified
if os.path.isdir(file_or_directory):
    files_to_process = scan_directory(file_or_directory, extensions)
elif os.path.isfile(file_or_directory):
    files_to_process = [file_or_directory]
else:
    files_to_process = []
    print("Invalid path. Please specify a valid file or directory.")

print(f"Found {len(files_to_process)} file(s) to process.")


Found 368 file(s) to process.


<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\ilo_l\AppData\Local\Temp\ipykernel_6748\2076886340.py:2: SyntaxWarning: invalid escape sequence '\M'
  file_or_directory = "H:\My Drive\Books"


In [3]:
import os
import shutil
from PyPDF2 import PdfReader, PdfWriter
from ebooklib import epub
import subprocess


def scan_directory(directory, extensions):
    """
    Scan a directory and return a list of files with the given extensions.
    """
    files = []
    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            if any(filename.lower().endswith(ext) for ext in extensions):
                files.append(os.path.join(root, filename))
    return files


def process_pdf(file_path):
    """
    Extract and clean metadata for a PDF file.
    """
    try:
        reader = PdfReader(file_path)
        writer = PdfWriter()
        
        # Extract metadata
        metadata = reader.metadata
        print(f"Original metadata for {file_path}: {metadata}")
        
        # Clean metadata
        cleaned_metadata = {}
        for key, value in metadata.items():
            if isinstance(value, str) and len(value) < 1000:  # Filter unusually long entries
                cleaned_metadata[key] = value
        
        # Write the new metadata to a new PDF
        for page in reader.pages:
            writer.add_page(page)
        writer.add_metadata(cleaned_metadata)
        
        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original
        with open(file_path, "wb") as f:
            writer.write(f)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def process_epub(file_path):
    """
    Extract and clean metadata for an EPUB file.
    """
    try:
        book = epub.read_epub(file_path)
        metadata = book.get_metadata("DC", "*")
        print(f"Original metadata for {file_path}: {metadata}")
        
        # Clean metadata
        cleaned_metadata = []
        for meta in metadata:
            key, value, _ = meta
            if isinstance(value, str) and len(value) < 1000:  # Filter unusually long entries
                cleaned_metadata.append((key, value, {}))
        
        # Set cleaned metadata
        book.set_metadata("DC", cleaned_metadata)
        
        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original
        epub.write_epub(file_path, book)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def process_other_files_with_exiftool(file_path):
    """
    Use ExifTool to clean metadata for other file types.
    """
    try:
        backup_path = file_path + ".backup"
        shutil.copy(file_path, backup_path)  # Backup original
        
        # Clean metadata using exiftool
        subprocess.run(["exiftool", "-all=", "-overwrite_original", file_path], check=True)
        print(f"Metadata cleaned for {file_path}. Backup saved at {backup_path}.")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
